# 02 — Candidate Claims, Source Evidence, and Factuality Signals

**Purpose:** split each saved hierarchy output into sentence-level candidate claims, retrieve source-text windows, and score the claim/evidence pairs with a pinned natural-language-inference (NLI) classifier.

This stage does **not** regenerate NCM outputs. Evidence-pair scores are cached in `nli_pair_cache.sqlite3` keyed by claim, evidence window, and model revision. If the runtime disconnects, rerunning the cell reuses scores already committed.

**Interpretation:** `supported`, `contradicted`, and `unresolved` are automated evidence labels, not human-adjudicated truth. Sentence boundaries do not guarantee atomic claims. Thresholds are fixed in the configuration before seeing results.

In [ ]:
# 0.2 — Annotation stage
from pathlib import Path
import sys
import subprocess
import importlib

PROJECT_PARENT = Path("/content/drive/MyDrive/NCM Research")
PROJECT_ROOT = PROJECT_PARENT / "github"
SRC_ROOT = PROJECT_ROOT / "src"

# Ensure Google Drive is mounted.
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

# Validate the repository and source layout.
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Repository not found: {PROJECT_ROOT}\n"
        "Expected the repository at MyDrive/NCM Research/github."
    )

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError(
        f"pyproject.toml not found in {PROJECT_ROOT}. "
        "The repository may be incomplete."
    )

if not (PROJECT_ROOT / "configs" / "experiment.yaml").is_file():
    raise FileNotFoundError(
        f"Experiment config not found: "
        f"{PROJECT_ROOT / 'configs' / 'experiment.yaml'}"
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")

# Install/re-register the package from the actual repository.
# This does not rerun NCM inference.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps", "--no-build-isolation", "-e", str(PROJECT_ROOT)
    ],
    check=True,
)

# Ensure this notebook resolves the local source tree.
sys.path.insert(0, str(SRC_ROOT))
importlib.invalidate_caches()

# Import and execute annotation.
from ncm_factuality.config import load_config, resolve_paths
from ncm_factuality.annotation import run_annotation

cfg = load_config(PROJECT_ROOT / "configs" / "experiment.yaml")
paths = resolve_paths(PROJECT_ROOT, cfg)

print("Starting annotation...")
annotation_manifest = run_annotation(cfg, paths)

print("Annotation stage completed.")
annotation_manifest

## Output contract

- `claims.jsonl`: candidate sentence claims derived from saved hierarchy outputs.
- `claims_annotated.jsonl`: candidate claims with model score summaries, label and annotation provenance.
- `nli_pair_cache.sqlite3`: evidence-pair score cache for lossless resume.
- `annotation_manifest.json`: number of claim instances, unique keys, pair scores and label counts.

The source documents stay private on Drive. Later export strips full article and reference text, evidence-window contents, and local absolute paths.